# DeepDream — PyTorch edition

A PyTorch translation of the [TensorFlow DeepDream tutorial](https://www.tensorflow.org/tutorials/generative/deepdream)
(the ancestor of this notebook — portions adapted from work © 2019 The TensorFlow
Authors, Apache License 2.0), implementing "Inceptionism"
([Mordvintsev et al., 2015](https://ai.googleblog.com/2015/06/inceptionism-going-deeper-into-neural.html)):
forward an image through a CNN, take the gradient of chosen layer activations
with respect to the image, and repeatedly add it back so the image increasingly
excites those layers.

**Environment** — this notebook targets the `deepdream` conda env (own env, *not*
`fractal` — see `environment.yml` next to this notebook for the why of every pin):

```bash
conda env create -f environment.yml
conda activate deepdream
python -m ipykernel install --user --name deepdream --display-name "Python (deepdream)"
```

Pinned to `torch 2.7.1+cu118` — the **final PyTorch release with CUDA 11.8
builds**, and therefore the last whose wheels ship **sm_50 Maxwell kernels**
(they run on the Titan X sm_52 cards via same-major compatibility). Targets the
Ubuntu 22.04 machines; the wheels are manylinux_2_28, so they will *not* install
on the 18.04 laptop (use `torch==2.6.0+cu118` there — same notebook, one pin
back). Everything below is eager fp32 — the right lane for Maxwell. First model
use downloads ~104 MB of InceptionV3 weights to `~/.cache/torch`.

In [ ]:
import numpy as np
import PIL.Image
import IPython.display as display

import torch
import torch.nn.functional as F
import torchvision
from torchvision.models import inception_v3, Inception_V3_Weights
from torchvision.models.feature_extraction import create_feature_extractor

torch.__version__, torchvision.__version__, torch.version.cuda

In [ ]:
# The workstation has 4x Titan X: pick a card with CUDA_VISIBLE_DEVICES=n before
# launching Jupyter, or change the index here. A single dream stays on one card
# (multi-GPU frame sharding is repo milestone 6, and it happens per frame, not here).
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    cap = torch.cuda.get_device_capability(device)
    print(torch.cuda.get_device_name(device), "— compute capability", cap)
    print("kernels in this torch build:", torch.cuda.get_arch_list())
    # Titan X (Maxwell) reports (5, 2): it runs the sm_50 kernels above.
    # Maxwell lane: fp32 only (fp64 is 1/32 speed, no fast fp16) and no
    # torch.compile (Triton needs sm_70+) — this notebook is eager fp32 throughout.
else:
    print("No CUDA device visible — running on CPU (works, but slowly).")

## Choose an image to dream-ify

The stock tutorial image is a
[labrador](https://commons.wikimedia.org/wiki/File:YellowLabradorLooking_new.jpg);
the cell after it prefers a local image (e.g. a Fractal Studio render) and falls
back to the labrador if the path isn't on this machine.

In [ ]:
import pathlib, urllib.request

CACHE = pathlib.Path.home() / ".cache" / "deepdream"
CACHE.mkdir(parents=True, exist_ok=True)

url = 'https://storage.googleapis.com/download.tensorflow.org/example_images/YellowLabradorLooking_new.jpg'

# Download an image and read it into a NumPy array (stand-in for tf.keras.utils.get_file).
def download(url, max_dim=None):
    path = CACHE / url.split('/')[-1]
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    img = PIL.Image.open(path)
    if max_dim:
        img.thumbnail((max_dim, max_dim))
    return np.array(img)

# Undo model-space scaling: float [-1, 1] -> uint8 [0, 255] on the CPU.
def deprocess(img):
    img = 255 * (img + 1.0) / 2.0
    return img.clamp(0, 255).to(torch.uint8).cpu().numpy()

# Display an image
def show(img):
    if isinstance(img, torch.Tensor):
        img = img.cpu().numpy()
    display.display(PIL.Image.fromarray(np.asarray(img)))

In [ ]:
IMAGE_PATH = '/home/rafa/Pictures/FractalFlowerDream2.jpg'   # e.g. a Fractal Studio render

try:
    img = np.array(PIL.Image.open(IMAGE_PATH))
except FileNotFoundError:
    print('Local image not found — falling back to the tutorial labrador.')
    img = download(url, max_dim=500)

img = img[:, :, :3]          # drop an alpha channel if present
img.shape, img.min(), img.max()

## Prepare the feature extraction model

Same model as the original: **InceptionV3** pretrained on ImageNet. torchvision's
`IMAGENET1K_V1` checkpoint and Keras's `imagenet` weights both descend from the
original TF-slim training run, so the dream character carries over (not
bit-identical — porting details differ).

Two translation details:

* torchvision's pretrained InceptionV3 normally expects ImageNet-normalized input
  and internally rescales it (`transform_input=True`). Setting
  `transform_input = False` makes the network consume TF-convention input
  directly — `x / 127.5 - 1`, exactly what
  `tf.keras.applications.inception_v3.preprocess_input` produced — so the
  tutorial's `[-1, 1]` clipping and `deprocess` carry over unchanged.
* Keras names the 11 filter-concatenation layers `mixed0`…`mixed10`; torchvision
  names the same modules `Mixed_5b`…`Mixed_7c`. The dict below maps them, so
  layer picks keep their familiar Keras names:

| Keras | torchvision | | Keras | torchvision | | Keras | torchvision |
|---|---|---|---|---|---|---|---|
| mixed0 | Mixed_5b | | mixed4 | Mixed_6b | | mixed8 | Mixed_7a |
| mixed1 | Mixed_5c | | mixed5 | Mixed_6c | | mixed9 | Mixed_7b |
| mixed2 | Mixed_5d | | mixed6 | Mixed_6d | | mixed10 | Mixed_7c |
| mixed3 | Mixed_6a | | mixed7 | Mixed_6e | | | |

Lower layers dream strokes and textures; deeper layers dream objects (eyes, dogs).
`create_feature_extractor` is the direct analog of
`tf.keras.Model(inputs, outputs=layers)` — it prunes the graph after the deepest
requested layer, and accepts any input size the strided stem can digest
(keep every octave ≳ 100 px per side).

In [ ]:
weights = Inception_V3_Weights.IMAGENET1K_V1
base_model = inception_v3(weights=weights)
base_model.transform_input = False    # consume TF-style [-1, 1] input directly
base_model.eval().requires_grad_(False).to(device)

# Keras InceptionV3 concat-layer names -> torchvision module names
KERAS_TO_TV = {
    'mixed0': 'Mixed_5b', 'mixed1': 'Mixed_5c', 'mixed2': 'Mixed_5d',
    'mixed3': 'Mixed_6a', 'mixed4': 'Mixed_6b', 'mixed5': 'Mixed_6c',
    'mixed6': 'Mixed_6d', 'mixed7': 'Mixed_6e', 'mixed8': 'Mixed_7a',
    'mixed9': 'Mixed_7b', 'mixed10': 'Mixed_7c',
}

# Maximize the activations of these layers
names = ['mixed7', 'mixed9']   # the stock tutorial uses ['mixed3', 'mixed5']

# Create the feature extraction model
dream_model = create_feature_extractor(
    base_model, return_nodes={KERAS_TO_TV[n]: n for n in names}
)

## Calculate loss

The loss is the sum of the mean activation of each chosen layer (the per-layer
mean keeps big layers from drowning out small ones). Normally you'd minimize a
loss; DeepDream *maximizes* it via gradient ascent.

In [ ]:
def calc_loss(img, model):
    # img is float32, HWC, in [-1, 1]. Make it an NCHW batch of one.
    img_batch = img.permute(2, 0, 1).unsqueeze(0)
    layer_activations = model(img_batch)              # dict: name -> activation
    losses = [act.mean() for act in layer_activations.values()]
    return sum(losses)

## Gradient ascent

Compute the gradient of the loss with respect to the image, normalize it, and add
it to the image — each step makes the image excite the chosen layers a little
more. `torch.autograd.grad` plays the role of `tape.gradient`.

The TF version wrapped this in `@tf.function` for speed. The torch equivalent
(`torch.compile`) is off the table on the Titan X — its Triton backend needs
sm_70+ — but eager is fine here: the Inception forward/backward dominates, and
the Python loop overhead is noise. (On the GTX 1650 you can experiment with
`torch.compile` if curious.)

In [ ]:
class DeepDream:
    def __init__(self, model):
        self.model = model

    def __call__(self, img, steps, step_size):
        loss = torch.tensor(0.0)
        for n in range(steps):
            # A fresh leaf tensor each step, so gradients flow to the pixels.
            img = img.detach().requires_grad_(True)
            loss = calc_loss(img, self.model)

            # Gradient of the loss with respect to the pixels of the input image.
            gradients = torch.autograd.grad(loss, img)[0]

            # Normalize the gradients.
            gradients = gradients / (gradients.std() + 1e-8)

            # Gradient *ascent*: add the gradients so the image increasingly
            # "excites" the layers, then keep it in the model's [-1, 1] range.
            with torch.no_grad():
                img = (img + gradients * step_size).clamp(-1.0, 1.0)

        return loss, img.detach()

In [ ]:
deepdream = DeepDream(dream_model)

## Main Loop

In [ ]:
def preprocess(img):
    # uint8/float [0, 255] array -> float32 [-1, 1] tensor on the device.
    # (Same convention as tf.keras.applications.inception_v3.preprocess_input.)
    img = torch.as_tensor(np.asarray(img), dtype=torch.float32, device=device)
    return img / 127.5 - 1.0

def run_deep_dream_simple(img, steps=100, step_size=0.01):
    img = preprocess(img)
    steps_remaining = steps
    step = 0
    while steps_remaining:
        # Chunks of <=100 steps: in TF this limited retracing; here it just
        # sets the progress-display cadence.
        run_steps = min(100, steps_remaining)
        steps_remaining -= run_steps
        step += run_steps

        loss, img = deepdream(img, run_steps, step_size)

        display.clear_output(wait=True)
        show(deprocess(img))
        print(f'Step {step}, loss {loss.item():.4f}')

    result = deprocess(img)
    display.clear_output(wait=True)
    show(result)
    return result

In [ ]:
dream_img = run_deep_dream_simple(img=img, steps=150, step_size=0.01)

## Taking it up an octave

The single-scale result is noisy, low-resolution, and single-granularity. The
fix: run gradient ascent, scale the image up (an *octave*), and repeat — patterns
grown at small scales get elaborated with detail at larger ones.

In [ ]:
import time
start = time.time()

OCTAVE_SCALE = 1.30

def resize_img(img, size):
    # HWC (numpy or tensor, any value range) -> bilinear resize on the device -> float32 HWC
    t = torch.as_tensor(np.asarray(img) if not isinstance(img, torch.Tensor) else img,
                        dtype=torch.float32, device=device)
    t = t.permute(2, 0, 1).unsqueeze(0)
    t = F.interpolate(t, size=tuple(int(s) for s in size),
                      mode='bilinear', align_corners=False)
    return t.squeeze(0).permute(1, 2, 0)

# Work on a copy so `img` stays the pristine base image
# (the TF notebook rebinds `img` here, which gets confusing two cells later).
octave_img = np.asarray(img)
base_shape = octave_img.shape[:-1]

for n in range(-2, 3):
    new_shape = tuple(int(round(s * OCTAVE_SCALE ** n)) for s in base_shape)
    octave_img = resize_img(octave_img, new_shape).cpu().numpy()
    octave_img = run_deep_dream_simple(img=octave_img, steps=50, step_size=0.01)

display.clear_output(wait=True)
octave_result = resize_img(octave_img, base_shape).clamp(0, 255).to(torch.uint8)
show(octave_result)

print(f'{time.time() - start:.1f} s')

## Optional: Scaling up with tiles

As the image grows, so does the memory and time for one whole-image gradient.
Splitting the image into tiles and summing per-tile gradients keeps memory flat
at any resolution (12 GB on the Titan X, 4 GB on the 1650 — drop `tile_size` to
256 there if you hit OOM).

A random shift before each tiled pass stops tile seams from printing into the
image. Start with the random shift:

In [ ]:
def random_roll(img, maxroll):
    # Randomly shift the image to avoid tiled boundaries.
    shift = torch.randint(-maxroll, maxroll, (2,))
    img_rolled = torch.roll(img, shifts=(int(shift[0]), int(shift[1])), dims=(0, 1))
    return shift, img_rolled

In [ ]:
shift, img_rolled = random_roll(torch.as_tensor(np.asarray(img)), 512)
show(img_rolled)

Here is a tiled equivalent of the `deepdream` function defined earlier.
`loss.backward()` *accumulates* each tile's gradient into `img_rolled.grad` —
the running-sum bookkeeping the TF version carried by hand:

In [ ]:
class TiledGradients:
    def __init__(self, model):
        self.model = model

    def __call__(self, img, tile_size=512):
        shift, img_rolled = random_roll(img, tile_size)
        img_rolled = img_rolled.detach().requires_grad_(True)

        h, w = img_rolled.shape[:2]
        # Skip the last (ragged) tile, unless there's only one tile.
        # The random roll ensures every pixel is still visited across steps.
        xs = list(range(0, w, tile_size))[:-1] or [0]
        ys = list(range(0, h, tile_size))[:-1] or [0]

        for x in xs:
            for y in ys:
                # Extract a tile and accumulate its gradient into img_rolled.grad.
                img_tile = img_rolled[y:y + tile_size, x:x + tile_size]
                loss = calc_loss(img_tile, self.model)
                loss.backward()

        # Undo the random shift applied to the image and its gradients.
        gradients = torch.roll(img_rolled.grad,
                               shifts=(-int(shift[0]), -int(shift[1])), dims=(0, 1))

        # Normalize the gradients.
        gradients = gradients / (gradients.std() + 1e-8)
        return gradients

In [ ]:
get_tiled_gradients = TiledGradients(dream_model)

Putting this together gives a scalable, octave-aware deepdream implementation:

In [ ]:
def run_deep_dream_with_octaves(img, steps_per_octave=100, step_size=0.01,
                                octaves=range(-2, 3), octave_scale=1.3):
    base_shape = np.asarray(img).shape[:-1]
    img = preprocess(img)

    for octave in octaves:
        # Scale the image based on the octave
        new_size = tuple(int(round(s * octave_scale ** octave)) for s in base_shape)
        img = resize_img(img, new_size)

        for step in range(steps_per_octave):
            gradients = get_tiled_gradients(img)
            with torch.no_grad():
                img = (img + gradients * step_size).clamp(-1.0, 1.0)

            if step % 10 == 0:
                display.clear_output(wait=True)
                show(deprocess(img))
                print(f'Octave {octave}, Step {step}')

    return deprocess(img)

In [ ]:
dream_big = run_deep_dream_with_octaves(img=img, step_size=0.01)

display.clear_output(wait=True)
final = resize_img(dream_big, np.asarray(img).shape[:-1]).clamp(0, 255).to(torch.uint8)
show(final)

Much better! Play with the octave count, octave scale, and layer picks to change
the character of the dream.

---

**Everything below is new** — not in the TF tutorial, added because it maps onto
this repo's plan (`CLAUDE.md`).

### Save the result

In [ ]:
out_dir = pathlib.Path('out')
out_dir.mkdir(exist_ok=True)
PIL.Image.fromarray(final.cpu().numpy()).save(out_dir / 'dream.png')
print('saved', out_dir / 'dream.png')

### Optional: dream-zoom video

The classic Inceptionism feedback loop: dream a little, zoom in a little, feed the
result back in. Seeding each frame from the previous dreamed frame is the
simplest form of the repo's `coherence/` idea (milestone 3) — it's what kills
flicker. The writer uses the exact H.264 settings validated in Fractal Studio's
video pipeline (CRF, not fixed-quality). Keep the source ≲ 1080p for the untiled
per-frame dream.

In [ ]:
import imageio

def zoom(frame, factor):
    # Scale about the center by `factor`, crop back to the original size.
    h, w = frame.shape[:2]
    big = resize_img(frame, (int(h * factor), int(w * factor)))
    y0 = (big.shape[0] - h) // 2
    x0 = (big.shape[1] - w) // 2
    return big[y0:y0 + h, x0:x0 + w]

def dream_zoom_video(img, path='out/dream_zoom.mp4', n_frames=90, fps=30,
                     zoom_per_frame=1.01, steps_per_frame=8, step_size=0.01):
    frame = preprocess(img)
    h, w = frame.shape[:2]
    frame = frame[:h // 2 * 2, :w // 2 * 2]          # yuv420p needs even dims

    writer = imageio.get_writer(path, fps=fps, codec='libx264',
                                pixelformat='yuv420p',
                                output_params=['-crf', '20', '-preset', 'slow'])
    try:
        for i in range(n_frames):
            _, frame = deepdream(frame, steps_per_frame, step_size)
            writer.append_data(deprocess(frame))
            frame = zoom(frame, zoom_per_frame).clamp(-1.0, 1.0)
            if i % 10 == 0:
                print(f'frame {i}/{n_frames}')
    finally:
        writer.close()
    print('wrote', path)

# dream_zoom_video(img)    # uncomment to render (a few minutes on the Titan X)

### Toward the real objective: weighted `(layer, channel, weight)` targets

The repo's design commitment #2 is dreaming toward a weighted combination of
feature channels, negative weights suppressing a feature. Everything below the
model boundary speaks *preprocessed tensors* (`float32`, `[-1, 1]`, on the
device) — `preprocess()` is the door in, `deprocess()` the door out. Channel
indices are the thing to *discover* with contact sheets (milestone 2); at these
taps, `mixed7` has 768 channels and `mixed9` has 2048.

In [ ]:
def make_target_extractor(targets):
    # An extractor covering exactly the layers the targets mention, so targets
    # may roam anywhere in mixed0..mixed10 regardless of `names` above.
    layers = sorted({layer for layer, _, _ in targets})
    return_nodes = {KERAS_TO_TV[name]: name for name in layers}
    return create_feature_extractor(base_model, return_nodes=return_nodes).to(device).eval()

def calc_loss_targets(img, model, targets):
    # img: a *preprocessed* tensor (HWC, [-1, 1]) like every loss here eats.
    # A raw numpy image is converted as a courtesy for one-off probing, but
    # inside gradient loops always pass the tensor.
    if not torch.is_tensor(img):
        img = preprocess(img)
    acts = model(img.permute(2, 0, 1).unsqueeze(0))
    return sum(w * acts[layer][0, channel].mean() for layer, channel, w in targets)

class TargetDream:
    # DeepDream, but the objective is a weighted shopping list of channels.
    def __init__(self, targets):
        self.targets = targets
        self.model = make_target_extractor(targets)

    def __call__(self, img, steps, step_size):
        loss = torch.tensor(0.0)
        for n in range(steps):
            img = img.detach().requires_grad_(True)
            loss = calc_loss_targets(img, self.model, self.targets)
            gradients = torch.autograd.grad(loss, img)[0]
            gradients = gradients / (gradients.std() + 1e-8)
            with torch.no_grad():
                img = (img + gradients * step_size).clamp(-1.0, 1.0)
        return loss, img.detach()

def run_deep_dream_targets(img, targets, steps=100, step_size=0.01):
    dreamer = TargetDream(targets)
    if not torch.is_tensor(img):
        img = preprocess(img)
    steps_remaining, step = steps, 0
    while steps_remaining:
        run_steps = min(100, steps_remaining)
        steps_remaining -= run_steps
        step += run_steps
        loss, img = dreamer(img, run_steps, step_size)
        display.clear_output(wait=True)
        show(deprocess(img))
        print(f'Step {step}, loss {loss.item():.4f}')
    result = deprocess(img)
    display.clear_output(wait=True)
    show(result)
    return result

# Amplify two mixed7 channels, damp one mixed9 channel:
targets = [('mixed7', 10, 1.0), ('mixed7', 99, 0.7), ('mixed9', 4, -0.5)]
targeted_img = run_deep_dream_targets(img, targets, steps=100, step_size=0.01)